# 🤖 Module 1 — Nhập môn Lập trình Ứng dụng LLM

In [ ]:
# --- Setup: chạy cell này một lần duy nhất ---
# Cài đặt thư viện (bỏ comment nếu bạn chạy lần đầu):
# !pip install -U transformers accelerate matplotlib seaborn
import os
import time
import random
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import logging
import sys

sys.stderr = open(os.devnull, 'w')
logging.set_verbosity_error()
torch.set_num_threads(6)

In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer
from transformers.pipelines import pipeline
MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
print(f"Loading {MODEL_NAME} on CPU ... (lần đầu có thể mất 1–2 phút)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cpu",
    torch_dtype="auto",
    trust_remote_code=False,
    attn_implementation="sdpa" 
)

model = torch.compile(model)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)
print("✅ Model sẵn sàng!")

---

## 🛠️ Bài tập thực hành



### Bài tập 1 — Hallucination Detector

#### 🎯 Mục tiêu
- Thiết kế một "detector" đơn giản: hỏi LLM, rồi kiểm tra câu trả lời có chứa keyword đúng không.
- Trực tiếp trải nghiệm LLM **tự tin trả lời sai** với câu hỏi không có đáp án thực.

#### 📝 Hướng dẫn thực hiện
**TODO** — Duyệt qua tất cả expected_keywords nếu có bất kỳ keyword nào trong answer thì trả ra False còn không ra True.


In [ ]:
def ask_llm(question: str) -> str:
    out = generator(question, max_new_tokens=30,do_sample=False)[0]["generated_text"]
    return out.strip()

def is_hallucination(answer: str, expected_keywords: list) -> bool:
    """True nếu answer không chứa keyword nào, nếu chứa thì False.
        Bước 1: Duyệt qua tất cả keyword trong expected_keywords
        Bước 2: Kiểm tra xem có bất kỳ keyword trong expected_keywords nào xuất hiện trong answer không, 
                nếu có return False (không bị ảo giác) nếu không return True 
    """



    # TODO: trả True nếu KHÔNG có keyword nào xuất hiện trong answer, ngược lại ra False
    return 

In [ ]:
# Kiểm tra kết quả

# 3 câu: 1 fact thật + 2 bịa
questions = [
    ("Thủ đô của Việt Nam là gì? Trả lời ngắn gọn trong 1 câu",              ["hà nội", "hanoi"]),
    ("Việt nam có bao nhiêu đầu đạn hạt nhân? Trả lời ngắn gọn trong 1 câu", ['không biết','không','chưa']),
    ("Ai phát minh máy tính lượng tử năm 1850? Trả lời ngắn gọn trong 1 câu",['không biết','không','chưa']),
]  # Giá trị của list là các tuple gồm 2 phần : (question, expected_keywords)


# Duyệt qua  (question, expected_keywords) trong list -> gọi llm trả lời question
#            -> kiểm tra hallucination
for q, kws in questions:
    ans = ask_llm(q)
    is_h = is_hallucination(ans, kws)
    tag = "🚨 Hallucination" if is_h else "✅ OK"
    print(f"[{tag}]  Q: {q}")
    print(f"         A: {ans}\n")

### Bài tập 2 — Sentiment Analyzer bằng Prompt

#### 🎯 Mục tiêu
- Dùng LLM như một **classifier không cần training**: chỉ cần viết prompt đúng.
- Biết cách thiết kế prompt để LLM trả về output có cấu trúc (1 nhãn duy nhất).


#### 📝 Hướng dẫn thực hiện
0. Chuẩn bị một model mới mạnh hơn là **Qwen/Qwen3.5-0.8B**
1. **TODO 2a** — Xây dựng `system prompt` và `prompt` hướng dẫn LLM phân loại thành `'tích cực'`, `'tiêu cực'`, hoặc `'trung lập'`, **chỉ trả 1 cụm, không giải thích**.
2. **TODO 2b** — Kiểm tra label có nằm trong kết quả trả về của model hay không.


In [ ]:
from transformers import AutoModelForCausalLM,AutoTokenizer
from transformers.pipelines import pipeline
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
print(f"Loading {MODEL_NAME} on CPU ... (lần đầu có thể mất 1–2 phút)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cpu",
    torch_dtype="auto",
    trust_remote_code=False,
    attn_implementation="sdpa" 
)

model = torch.compile(model)
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
)
print("✅ Model sẵn sàng!")

In [ ]:
import re

def clear_reasoning(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    
def analyze_sentiment(text: str) -> str:
    """Dùng LLM phân loại cảm xúc: 'tích cực', 'tiêu cực', hoặc 'trung lập'.
        Bước 1: Thiết kế system prompt và prompt sao cho chỉ trả về 3 label trên.
        Bước 2: Chuẩn hóa label và ans thành câu viết thường .lower(). 
        Bước 3: Kiểm tra label có trong kết quả trả về của model hay không không và nêu có thì trả về label.

        Kiểm tra và tự đánh giá lại.
        system prompt: là chỉ thị cấp cao nhất dùng để định hướng cách LLM hành xử trong suốt cuộc hội thoại.
            - Ví dụ: system prompt = Bạn là 1 chuyên gia toán học và chỉ đưa ra kết quả mà không cần trình bày. 
    """

    # TODO 2a: Xây dựng system prompt và prompt yêu cầu LLM chỉ trả 1 cụm từ (không giải thích)
    system_prompt= ... 
    prompt = ...

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    ans = clear_reasoning(ask_llm(messages))

    # Trích xuất nhãn từ response (đã viết sẵn — không cần sửa)
    for label in ["tích cực", "tiêu cực", "trung lập"]:
         # TODO 2b: check label trong ans và return về label
        ... 
    return "không rõ"


In [ ]:
# Chạy thử 4 câu
samples = [
    "Sản phẩm này quá tệ.",
    "Hôm nay tôi rất vui vì vừa pass được bài tập khó!",
    "Hôm nay trời có mây, nhiệt độ 28 độ.",
    "Cây bút bi của bạn thì đang nằm trên bàn.",
    "Ứng dụng này siêu tiện lợi, tôi yêu nó!",
]
labels = []
for s in samples:
    label = analyze_sentiment(s)
    labels.append(label)
    print(f"{label:>12} | {s}")

---

## 🏁 Kết thúc Module 1

Chúc mừng! Đến đây bạn đã:

- ✅ Phân biệt **deterministic vs probabilistic** và hiểu đây là thay đổi tư duy lớn nhất khi chuyển sang lập trình với LLM.
- ✅ Hiểu cơ chế **dự đoán token tiếp theo**: text → logits → softmax → 1 token — và vòng lặp autoregressive.
- ✅ **Tự xây vòng lặp autoregressive** bằng Python thuần, không cần biết PyTorch.
- ✅ Trải nghiệm **hallucination** và có baseline detector đơn giản bằng keyword matching.
- ✅ Dùng LLM như **classifier không cần training** — chỉ qua prompt engineering.
